In [ ]:
import pandas as pd
import numpy as np
import os

# 3.1 Load structured datasets
raw_dir = "../data/raw"
pairs = ["EUR_USD", "USD_JPY", "GBP_USD", "AUD_USD", "USD_INR"]
data_dict = {}

for p in pairs:
    path = os.path.join(raw_dir, f"{p}.csv")
    if os.path.exists(path):
        data_dict[p] = pd.read_csv(path, parse_dates=['Date'])

print("--- DATA QUALITY CONTROL REPORT ---")
# 3.2 Data Quality and Completeness Checks
for name, df in data_dict.items():
    nulls = df.isnull().sum().sum()
    duplicates = df.duplicated(subset=['Date']).sum()
    
    # Calculate log returns to evaluate statistical parameters
    df['Returns'] = np.log(df['Close'] / df['Close'].shift(1))
    df = df.dropna()
    
    # Descriptive Statistics
    mean_ret = df['Returns'].mean()
    std_ret = df['Returns'].std()
    skew = df['Returns'].skew()
    kurt = df['Returns'].kurtosis() # High kurtosis flags tail risks/flash crashes
    
    print(f"\nAsset: {name}")
    print(f"  Rows: {len(df)} | Missing Fields: {nulls} | Duplicate Timestamps: {duplicates}")
    print(f"  Daily Return Moments -> Mean: {mean_ret:.6f} | Volatility: {std_ret:.6f}")
    print(f"  Asymmetry Details   -> Skewness: {skew:.4f} | Excess Kurtosis: {kurt:.4f}")

# 3.3 Dynamic Multi-Pair Cross-Correlation Matrix
close_prices = {}
for name, df in data_dict.items():
    # Re-index on date to ensure alignment across different asset schedules
    close_prices[name] = df.set_index('Date')['Close']

df_corr = pd.DataFrame(close_prices).pct_change().dropna()
correlation_matrix = df_corr.corr()

print("\n--- MULTI-PAIR CROSS-ASSET CORRELATION MATRIX ---")
print(correlation_matrix.round(4))